# Codebook pipeline

Builds entailment claims from the IHRA, Nexus, and JDA texts. Self-contained, it only reads `codebook/sources/*.txt` and writes `codebook.yml` and `codebook.txt` back into `codebook/`.

Runtime: Colab with a GPU.

Stages:
1. anchor each text to the authors' own items, with polarity and context
2. propositionize each item with a small seq2seq model
3. render each proposition as a claim in the authors' wording
4. validate each claim against its source item

In [ ]:
# setup
import subprocess, sys, os, json, re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "spacy", "lemminflect", "pyyaml", "sentencepiece", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_trf", "-q"], check=True)

In [ ]:
# load
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# repo
REPO = "/content/drive/MyDrive/contested-antisemitism"  # point at a drive copy if codebook/ is not pushed
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/jwrampino/contested-antisemitism", REPO], check=True)
CODEBOOK = f"{REPO}/codebook"
assert os.path.exists(f"{CODEBOOK}/sources"), "codebook/sources not found in REPO"

SPLIT = "model"  # "model" uses the propositionizer's split, "item" keeps each author item whole

# first and last paragraph of each text to keep
SPAN = {"ihra": ("“Antisemitism is", None),
        "nexus": ("Antisemitism consists", "The Nexus Task Force"),
        "jda": ("Antisemitism is discrimination", None)}
TITLES = {"ihra": "IHRA Working Definition of Antisemitism",
          "nexus": "The Nexus Document",
          "jda": "Jerusalem Declaration on Antisemitism"}

In [ ]:
# anchor to author items
MARK = re.compile(r"^(•|\d{1,2}\.|[A-Za-z]\.)\s+")

def paragraphs(doc):
    raw = open(f"{CODEBOOK}/sources/{doc}.txt", encoding="utf-8").read()
    return [re.sub(r"\s*\n\s*", " ", p).strip() for p in re.split(r"\n\s*\n", raw) if p.strip()]

def is_heading(p):
    return not p.endswith((".", "”", '"', ":")) and len(p.split()) <= 30

def anchor(doc):
    first, last = SPAN[doc]
    ps = paragraphs(doc)
    i0 = next(i for i, p in enumerate(ps) if p.startswith(first))
    i1 = next((i for i, p in enumerate(ps) if last and p.startswith(last)), len(ps))
    items, heading, lead, parent, pol, bullets, loose = [], None, None, None, None, 0, 0
    for p in ps[i0:i1]:
        m = MARK.match(p)
        if p.endswith(":") and not m:  # lead-in sentence, context for the items after it
            lead = p
            continue
        if is_heading(p):
            heading, lead = p, None
            if re.search(r"\bnot\b", p, re.I):
                pol = "no"
            elif m or p.endswith("?"):
                pol = "yes"
            continue
        if m:
            mk, body = m.group(1), p[m.end():].rstrip(":")
            sub = mk[0].islower()
            if mk == "\u2022":
                bullets += 1; num = f"b{bullets}"
            else:
                num = mk.rstrip(".")
            if sub:
                key_num, ctx = f"{parent_num}{num}", parent
            else:
                parent, parent_num, key_num, ctx = body, num, num, lead
            key = f"{doc}:{'N' if pol == 'no' else ''}{key_num}"
            items.append(dict(doc=doc, key=key, polarity=pol or "yes", section=heading, context=ctx, text=body))
        else:  # unmarked paragraph, definition or explanation text
            loose += 1
            items.append(dict(doc=doc, key=f"{doc}:t{loose}", polarity=None, section=heading, context=lead, text=p.strip("\u201c\u201d")))
            lead = None
    return items

units = [u for d in SPAN for u in anchor(d)]
print(f"{len(units)} author items")
print([u["key"] for u in units])

In [ ]:
# load propositionizer
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "chentong00/propositionizer-wiki-flan-t5-large"
tok = AutoTokenizer.from_pretrained(MODEL)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32  # t5 fp16 is unstable
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=dtype).to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

In [ ]:
# propositionize
from tqdm.auto import tqdm

def propositionize(u):
    # title and section give the model the context it was trained with
    section = (u["section"] or u["context"] or "").strip()
    prompt = f"Title: {TITLES[u['doc']]}. Section: {section}. Content: {u['text']}"
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=512)
    raw = tok.decode(out[0], skip_special_tokens=True)
    try:
        props = json.loads(raw)
    except Exception:
        props = re.findall(r'"((?:[^"\\]|\\.)*)"', raw)  # fall back to quoted strings
    return raw, [p.strip() for p in props if isinstance(p, str) and p.strip()]

# cache raw outputs so reruns resume
cache_path = f"{CODEBOOK}/propositions.json"
cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
for u in tqdm(units):
    if u["key"] not in cache:
        raw, props = propositionize(u)
        cache[u["key"]] = dict(raw=raw, props=props)
        json.dump(cache, open(cache_path, "w"), indent=1)
    u["props"] = cache[u["key"]]["props"] if SPLIT == "model" else [u["text"]]
print(sum(len(u["props"]) for u in units), "propositions")

In [ ]:
# render claims
EVAL_VERBS = {"deem", "regard", "consider", "view", "judge"}
import spacy
from lemminflect import getInflection

nlp = spacy.load("en_core_web_trf")
_cache = {}
def parse(t):
    if t not in _cache:
        _cache[t] = nlp(t)
    return _cache[t]

def lcfirst(s, propn):
    return s if propn else s[:1].lower() + s[1:]

def tidy(c):
    c = re.sub(r"\s+", " ", c).strip(" ,.")
    return re.sub(r"\s+([,;:)])", r"\1", c)

# illustrations go to metadata, they are not part of the claim
def split_illustrations(text):
    kids = []
    text = text.strip().rstrip(".")
    while True:
        m = re.search(r"\s*\((?:e\.g\.,?|for example,?|i\.e\.,?)\s+([^)]*)\)", text, re.I)
        if not m:
            break
        kids.append(m.group(0).strip(" ()")); text = text[:m.start()] + text[m.end():]
    m = re.search(r",?\s+(?:e\.g\.,|for example,|such as|including|whether in)\s+(.*)$", text, re.I)
    if m:
        kids.append(m.group(0).strip(" ,")); text = text[:m.start()]
    return text.strip(" ,"), kids

# "it is <adj> to <vp>" keeps the vp, a general syntactic wrapper
def unwrap(text):
    doc = parse(text)
    roots = [t for t in doc if t.dep_ == "ROOT"]
    if roots and any(c.dep_ == "expl" for c in roots[0].children):
        r = roots[0]
        for c in r.children:
            if c.dep_ in ("xcomp", "ccomp", "advcl") and c.i > r.i:
                to = next((x for x in c.children if x.dep_ == "aux" and x.text.lower() == "to"), None)
                if to is not None:
                    neg = any(x.dep_ == "neg" for x in r.children)
                    return text[doc[to.i + 1].idx:], neg
    return text, False

def inflect_verbs(text):
    doc = parse(text)
    r = next(t for t in doc if t.dep_ == "ROOT")
    verbs = [r] + [c for c in r.conjuncts if c.tag_ in ("VBG", "VB")]
    out, ok = text, True
    for v in sorted(verbs, key=lambda t: -t.idx):
        form = getInflection(v.lemma_, "VBZ")
        if not form:
            ok = False; continue
        out = out[:v.idx] + form[0] + out[v.idx + len(v.text):]
    return out, ok

def frame(text):
    doc = parse(text)
    roots = [t for t in doc if t.dep_ == "ROOT"]
    propn = doc[0].pos_ == "PROPN"
    if not roots:
        return "The text claims that " + lcfirst(text, propn), ["no_root"]
    r = roots[0]
    if r.tag_ in ("VBG", "VB") and not any(c.dep_ in ("nsubj", "nsubjpass", "expl") for c in r.children):  # gerund or bare infinitive
        vp, ok = inflect_verbs(text)
        return "The text " + lcfirst(vp, propn), ([] if ok else ["inflect_failed"])
    if any(c.dep_ in ("nsubj", "nsubjpass", "expl") for c in r.children):
        return "The text claims that " + lcfirst(text, propn), ["clause_frame"]
    if r.pos_ in ("NOUN", "PROPN"):
        return "The text contains " + lcfirst(text, propn), ["np_frame"]
    return "The text claims that " + lcfirst(text, propn), ["clause_frame"]

# verdict structures in any wording
def verdict_structure(claim):
    for t in parse(claim):
        if t.lemma_ == "be" and any(c.dep_ == "expl" for c in t.children) and any(c.dep_ in ("acomp", "attr") for c in t.children):
            return True
        if t.lemma_ in EVAL_VERBS and any(c.dep_ == "auxpass" for c in t.children):
            return True
    return False

def negated_verdict(text):
    doc = parse(text)
    if not verdict_structure(text):
        return False
    return any(t.dep_ == "neg" for t in doc)

def to_claim(prop, pol):
    text, neg = unwrap(prop)
    neg = neg or negated_verdict(text)
    text, kids = split_illustrations(text)
    if text.count("“") != text.count("”"):
        text = text.strip("“” ")  # stray quote from the item boundary
    text = tidy(text)
    if not text:
        return None
    claim, flags = frame(text)
    if not claim.endswith(("”", ".")):
        claim += "."
    return dict(claim=claim, polarity="no" if neg else pol, flags=flags, illustrations=kids)

for u in tqdm(units):
    seen, u["claims"] = set(), []
    for p in u["props"]:
        c = to_claim(p, u["polarity"])
        if c and c["claim"] not in seen:
            seen.add(c["claim"]); u["claims"].append(c)

In [ ]:
# validate
FRAME_LEMMAS = {"text", "contain", "claim"}
BLOCKING = {"fragment", "inflect_failed", "unbalanced_paren", "bad_start", "no_root"}

def lemmas(s):
    return {t.lemma_.lower() for t in parse(s) if t.is_alpha and not t.is_stop}

def check(claim, flags, src):
    out = list(flags)
    if not re.match(r"^The text [a-z]", claim):
        out.append("bad_start")
    if claim.count("(") != claim.count(")"):
        out.append("unbalanced_paren")
    body = claim.rstrip(".").split()
    if len(body) <= 3 or body[-1].lower() in ("for", "of", "to", "in", "on", "with", "by", "from", "at"):
        out.append("fragment")
    if verdict_structure(claim) or re.search(r"\bantisemit\w*", claim, re.I):
        out.append("verdict_wording")  # wrapper or verdict left in the claim
    missing = sorted(lemmas(claim) - src - FRAME_LEMMAS)
    if missing:
        out.append("not_in_source:" + ",".join(missing))
    return sorted(set(out))

def status(flags):
    if any(f.split(":")[0] in BLOCKING for f in flags):
        return "blocked"
    return "review" if flags else "ok"

for u in units:
    src = lemmas(u["text"])
    used = set()
    for c in u["claims"]:
        c["flags"] = check(c["claim"], c["flags"], src)
        c["status"] = status(c["flags"])
        used |= lemmas(c["claim"])
        for ill in c["illustrations"]:
            used |= lemmas(ill)
    u["coverage"] = round(len(used & src) / max(len(src), 1), 2)

import pandas as pd
df = pd.DataFrame([dict(key=u["key"], polarity=c["polarity"], status=c["status"], claim=c["claim"], flags=", ".join(c["flags"]))
                   for u in units for c in u["claims"]])
print(df["status"].value_counts().to_string())
print("items with no claims:", [u["key"] for u in units if not u["claims"]])
print("low coverage:", [(u["key"], u["coverage"]) for u in units if u["coverage"] < 0.6])

In [ ]:
# review
def render_text(u):
    lines = [f"{u['key']}  [{u['polarity'] or '-'}]  coverage {u['coverage']}", f"  source: {u['text']}"]
    for c in u["claims"]:
        mark = "" if c["status"] == "ok" else f"  <{c['status']}: {', '.join(c['flags'])}>"
        lines.append(f"  - {c['claim']}{mark}")
        for ill in c["illustrations"]:
            lines.append(f"      illustration: {ill}")
    return "\n".join(lines) + "\n"

text_out = "\n".join(render_text(u) for u in units)
print(text_out)

In [ ]:
# save
import yaml
from google.colab import files

with open(f"{CODEBOOK}/codebook.yml", "w", encoding="utf-8") as f:
    yaml.safe_dump(dict(meta=dict(split=SPLIT, propositionizer=MODEL, spacy=nlp.meta["name"] + " " + nlp.meta["version"]),
                        items=[dict(key=u["key"], doc=u["doc"], polarity=u["polarity"], section=u["section"],
                                    source=u["text"], coverage=u["coverage"], claims=u["claims"]) for u in units]),
                   f, sort_keys=False, allow_unicode=True, width=110)
open(f"{CODEBOOK}/codebook.txt", "w", encoding="utf-8").write(text_out)
for name in ("codebook.yml", "codebook.txt", "propositions.json"):
    files.download(f"{CODEBOOK}/{name}")